In [3]:

"""AI Promotion Optimizer - Enhanced Version"""

# !pip install gradio pandas plotly numpy
# Uncomment above line when running in Colab

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime, timedelta
import random
import calendar
import os
import glob

random.seed(42)
np.random.seed(42)

class PromotionOptimizationSystem:
    def __init__(self):
        self.items_df = None

        self.family_insights = {
            "GROCERY I": {
                "response": "High", "best_promo": "BOGO (Buy One Get One)",
                "peak_days": "Weekend", "uplift": 0.35,
                "timing_advantage": "Weekend shopping creates bulk buying opportunities",
                "customer_behavior": "Value-driven bulk purchases increase basket size",
                "strategic_value": "Strong repeat purchase driver with high basket impact"
            },
            "CLEANING": {
                "response": "Medium", "best_promo": "Volume Discount",
                "peak_days": "Month-end", "uplift": 0.25,
                "timing_advantage": "Month-end creates stock-up mentality",
                "customer_behavior": "Planned purchases align with household budgets",
                "strategic_value": "Inventory clearance opportunity"
            },
            "BEVERAGES": {
                "response": "Very High", "best_promo": "Flash Sale (24-48 hours)",
                "peak_days": "Weekend", "uplift": 0.45,
                "timing_advantage": "Weekend Special creates urgency",
                "customer_behavior": "BEVERAGES items perform well during Weekend",
                "strategic_value": "Builds customer traffic and increases basket size"
            },
            "DAIRY": {
                "response": "Medium", "best_promo": "Percentage Off (20-40%)",
                "peak_days": "Weekly", "uplift": 0.20,
                "timing_advantage": "Weekly shopping pattern drives regular visits",
                "customer_behavior": "Freshness priority drives frequent purchases",
                "strategic_value": "Essential category driving store visits"
            },
            "BREAD/BAKERY": {
                "response": "High", "best_promo": "BOGO (Buy One Get One)",
                "peak_days": "Daily", "uplift": 0.30,
                "timing_advantage": "Daily consumption creates consistent demand",
                "customer_behavior": "Convenience-focused daily purchases",
                "strategic_value": "Perishable urgency creates momentum"
            },
            "PERSONAL CARE": {
                "response": "Medium", "best_promo": "Bundle Deal",
                "peak_days": "Month-start", "uplift": 0.22,
                "timing_advantage": "Payday spending peak at month-start",
                "customer_behavior": "Brand loyalty strong with payday spending",
                "strategic_value": "High margin premium positioning"
            },
            "MEATS": {
                "response": "Very High", "best_promo": "Flash Sale (24-48 hours)",
                "peak_days": "Weekend", "uplift": 0.50,
                "timing_advantage": "Weekend Special creates urgency",
                "customer_behavior": "Weekend grilling and family meal planning",
                "strategic_value": "Premium category with significant basket value"
            },
            "AUTOMOTIVE": {
                "response": "Low", "best_promo": "Limited Time Offer",
                "peak_days": "Seasonal", "uplift": 0.15,
                "timing_advantage": "Seasonal needs drive purchases",
                "customer_behavior": "Need-based infrequent purchases",
                "strategic_value": "Margin protection opportunity"
            },
            "HOME CARE": {
                "response": "Medium", "best_promo": "Volume Discount",
                "peak_days": "Spring", "uplift": 0.23,
                "timing_advantage": "Spring cleaning drives seasonal demand",
                "customer_behavior": "Seasonal cleaning with bulk buying",
                "strategic_value": "Stock-up opportunity"
            },
            "PRODUCE": {
                "response": "High", "best_promo": "Flash Sale (24-48 hours)",
                "peak_days": "Daily", "uplift": 0.40,
                "timing_advantage": "Daily freshness creates urgency",
                "customer_behavior": "Health-conscious freshness priority",
                "strategic_value": "Perishable nature creates urgency"
            },
            "FROZEN FOODS": {
                "response": "Medium", "best_promo": "Bundle Deal",
                "peak_days": "Month-end", "uplift": 0.25,
                "timing_advantage": "Month-end meal prep focus",
                "customer_behavior": "Convenience-driven meal preparation",
                "strategic_value": "Long shelf life enables extended promotions"
            },
            "BEAUTY": {
                "response": "High", "best_promo": "Percentage Off (20-40%)",
                "peak_days": "Holiday", "uplift": 0.35,
                "timing_advantage": "Holiday gifting occasions",
                "customer_behavior": "Emotional purchases and gifting",
                "strategic_value": "High margin loyalty opportunity"
            },
            "LIQUOR,WINE,BEER": {
                "response": "High", "best_promo": "Volume Discount",
                "peak_days": "Weekend", "uplift": 0.38,
                "timing_advantage": "Weekend social occasions",
                "customer_behavior": "Party planning and social events",
                "strategic_value": "High-value event-driven sales"
            }
        }

        self.yearly_holidays = {
            1: [{"day": 1, "name": "New Year's Day"}, {"day": 15, "name": "Mid-January Sale"}],
            2: [{"day": 14, "name": "Valentine's Day"}, {"day": 17, "name": "President's Day"}],
            3: [{"day": 17, "name": "St. Patrick's Day"}, {"day": 20, "name": "Spring Equinox"}],
            4: [{"day": 1, "name": "April Fool's Day"}, {"day": 22, "name": "Earth Day"}],
            5: [{"day": 5, "name": "Cinco de Mayo"}, {"day": 26, "name": "Memorial Day"}],
            6: [{"day": 15, "name": "Father's Day"}, {"day": 21, "name": "Summer Solstice"}],
            7: [{"day": 4, "name": "Independence Day"}, {"day": 15, "name": "Mid-Summer Sale"}],
            8: [{"day": 1, "name": "Back to School"}, {"day": 15, "name": "Mid-August Clearance"}],
            9: [{"day": 1, "name": "Labor Day"}, {"day": 22, "name": "Fall Equinox"}],
            10: [{"day": 31, "name": "Halloween"}, {"day": 15, "name": "Fall Festival"}],
            11: [{"day": 11, "name": "Veterans Day"}, {"day": 27, "name": "Thanksgiving"}],
            12: [{"day": 25, "name": "Christmas"}, {"day": 31, "name": "New Year's Eve"}]
        }

        self.upcoming_occasions = self._generate_upcoming_occasions()

    def _generate_upcoming_occasions(self):
        today = datetime.now()
        occasions = []

        for i in range(90):
            future_date = today + timedelta(days=i)
            date_str = future_date.strftime("%Y-%m-%d")
            day_of_week = future_date.strftime("%A")

            if future_date.weekday() in [5, 6]:
                occasions.append({
                    "date": date_str,
                    "occasion": f"Weekend Special ({day_of_week})",
                    "type": "weekend",
                    "priority": "high"
                })

            last_day = calendar.monthrange(future_date.year, future_date.month)[1]
            if future_date.day >= last_day - 2:
                occasions.append({
                    "date": date_str,
                    "occasion": "Month-End Sale",
                    "type": "month_end",
                    "priority": "medium"
                })

            if future_date.day <= 3:
                occasions.append({
                    "date": date_str,
                    "occasion": "Month-Start Special (Payday)",
                    "type": "month_start",
                    "priority": "high"
                })

            month_holidays = self.yearly_holidays.get(future_date.month, [])
            for holiday in month_holidays:
                if future_date.day == holiday["day"]:
                    occasions.append({
                        "date": date_str,
                        "occasion": holiday["name"],
                        "type": "major_holiday",
                        "priority": "very_high"
                    })

        unique_occasions = []
        seen_dates = set()
        for occ in occasions:
            if occ["date"] not in seen_dates:
                unique_occasions.append(occ)
                seen_dates.add(occ["date"])

        return sorted(unique_occasions, key=lambda x: x["date"])[:30]

    def auto_load_csv(self):
        possible_paths = [
            '/content/items.csv',
            'items.csv',
            '/content/drive/MyDrive/items.csv',
            './data/items.csv',
        ]

        csv_files = glob.glob('/content/*.csv') + glob.glob('*.csv')
        possible_paths.extend(csv_files)

        for path in possible_paths:
            if os.path.exists(path):
                try:
                    self.items_df = pd.read_csv(path)
                    if 'item_nbr' in self.items_df.columns and 'family' in self.items_df.columns:
                        self.upcoming_occasions = self._generate_upcoming_occasions()
                        return f"✅ Successfully loaded {len(self.items_df)} items from {len(self.items_df['family'].unique())} categories!"
                except:
                    continue

        return "⚠️ No valid items.csv found. Please upload to /content/ directory."

# Initialize system
promo_system = PromotionOptimizationSystem()
initial_load_status = promo_system.auto_load_csv()
print(initial_load_status)

def get_detailed_recommendations(num_recommendations=5):
    """Generate enhanced recommendations with card-style layout"""
    try:
        if promo_system.items_df is None:
            return """
            <div style='background: #fff3cd; padding: 20px; border-radius: 8px; border-left: 4px solid #ffc107;'>
                <h3 style='color: #856404; margin: 0;'>⚠️ Data Not Loaded</h3>
                <p style='margin: 10px 0 0 0;'>Please upload items.csv to /content/ directory.</p>
            </div>
            """

        current_date = datetime.now().strftime("%B %d, %Y")

        html_output = f"""
        <div style='margin-bottom: 30px;'>
            <h2 style='color: #2c3e50; margin-bottom: 5px;'>🎯 Promotion Recommendations</h2>
            <p style='color: #7f8c8d; margin: 0;'>Generated: {current_date}</p>
        </div>
        """

        available_families = promo_system.items_df['family'].unique()
        priority_families = ['BEVERAGES', 'MEATS', 'GROCERY I', 'PRODUCE', 'BEAUTY', 'DAIRY', 'BREAD/BAKERY']

        selected_families = [f for f in priority_families if f in available_families]
        for family in available_families:
            if family not in selected_families and len(selected_families) < num_recommendations:
                selected_families.append(family)

        for i in range(min(num_recommendations, len(selected_families))):
            family = selected_families[i]
            family_items = promo_system.items_df[promo_system.items_df['family'] == family]

            if len(family_items) > 0:
                item = family_items.sample(1).iloc[0]
                family_info = promo_system.family_insights.get(family, {
                    'best_promo': 'Percentage Off (20-40%)',
                    'uplift': 0.25,
                    'response': 'Medium',
                    'peak_days': 'Weekend',
                    'timing_advantage': 'Strategic timing advantage',
                    'customer_behavior': 'Customer purchasing patterns',
                    'strategic_value': 'Strategic business value'
                })

                next_occasion = promo_system.upcoming_occasions[i % len(promo_system.upcoming_occasions)]
                occ_date = datetime.strptime(next_occasion['date'], "%Y-%m-%d")
                days_until = (occ_date - datetime.now()).days

                # Color coding based on response level
                response_colors = {
                    'Very High': '#e74c3c',
                    'High': '#3498db',
                    'Medium': '#f39c12',
                    'Low': '#95a5a6'
                }
                response_level = family_info.get('response', 'Medium')
                card_color = response_colors.get(response_level, '#3498db')

                html_output += f"""
                <div style='background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
                            border-radius: 12px; padding: 25px; margin-bottom: 20px;
                            box-shadow: 0 4px 6px rgba(0,0,0,0.1); color: white;'>

                    <div style='border-bottom: 2px solid rgba(255,255,255,0.2); padding-bottom: 15px; margin-bottom: 20px;'>
                        <h3 style='margin: 0; font-size: 24px;'>🎯 Recommendation {i+1}</h3>
                    </div>

                    <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
                        <div>
                            <p style='margin: 8px 0; font-size: 15px;'><strong>📦 Item Number:</strong> {item['item_nbr']}</p>
                            <p style='margin: 8px 0; font-size: 15px;'><strong>📂 Category:</strong> {item['family']}</p>
                            <p style='margin: 8px 0; font-size: 15px;'><strong>🏷️ Promotion Type:</strong> {family_info.get('best_promo', 'N/A')}</p>
                        </div>
                        <div>
                            <p style='margin: 8px 0; font-size: 15px;'><strong>📅 Recommended Date:</strong> {next_occasion['date']} ({next_occasion['occasion']})</p>
                            <p style='margin: 8px 0; font-size: 15px;'><strong>📊 Predicted Uplift:</strong> <span style='background: rgba(255,255,255,0.2); padding: 2px 8px; border-radius: 4px; font-weight: bold;'>{family_info.get('uplift', 0.25):.0%}</span></p>
                        </div>
                    </div>

                    <div style='background: rgba(0,0,0,0.2); border-radius: 8px; padding: 15px; margin-bottom: 15px;'>
                        <h4 style='margin: 0 0 10px 0; color: #fff;'>🎯 Why This Promotion?</h4>
                        <p style='margin: 5px 0; font-size: 14px;'><strong>○ Category Response:</strong> {response_level} response rate</p>
                        <p style='margin: 5px 0; font-size: 14px;'><strong>○ Timing Advantage:</strong> {family_info.get('timing_advantage', 'N/A')}</p>
                        <p style='margin: 5px 0; font-size: 14px;'><strong>○ Customer Behavior:</strong> {family_info.get('customer_behavior', 'N/A')}</p>
                        <p style='margin: 5px 0; font-size: 14px;'><strong>○ Strategic Value:</strong> {family_info.get('strategic_value', 'N/A')}</p>
                    </div>

                    <div style='background: rgba(255,255,255,0.1); border-radius: 8px; padding: 15px;'>
                        <h4 style='margin: 0 0 10px 0; color: #fff;'>💡 Implementation Tips:</h4>
                        <p style='margin: 5px 0; font-size: 14px;'>○ Start promotion on {next_occasion['date']}</p>
                        <p style='margin: 5px 0; font-size: 14px;'>○ Combine with high-visibility store placement</p>
                        <p style='margin: 5px 0; font-size: 14px;'>○ Consider cross-merchandising with complementary items</p>
                        <p style='margin: 5px 0; font-size: 14px;'>○ Monitor stock levels - expect {family_info.get('uplift', 0.25):.0%} sales increase</p>
                    </div>
                </div>
                """

        return html_output

    except Exception as e:
        return f"""
        <div style='background: #f8d7da; padding: 20px; border-radius: 8px; border-left: 4px solid #dc3545;'>
            <h3 style='color: #721c24; margin: 0;'>❌ Error</h3>
            <p style='margin: 10px 0 0 0;'>{str(e)}</p>
        </div>
        """

def create_analysis_dashboard():
    """Create analysis visualizations"""
    try:
        if promo_system.items_df is None:
            return None, None, None, "❌ Data not loaded. Please upload items.csv."

        current_date = datetime.now().strftime("%B %d, %Y")

        # Chart 1: Category Distribution
        family_counts = promo_system.items_df['family'].value_counts().head(12)

        fig1 = px.bar(
            x=family_counts.values,
            y=family_counts.index,
            orientation='h',
            title="Product Portfolio Distribution",
            labels={'x': 'SKUs', 'y': 'Category'},
            color=family_counts.values,
            color_continuous_scale='Blues'
        )
        fig1.update_layout(
            height=500,
            showlegend=False,
            title_font_size=16,
            font=dict(size=11)
        )

        # Chart 2: ROI Matrix
        response_data = []
        for family in family_counts.index:
            info = promo_system.family_insights.get(family, {'uplift': 0.25, 'response': 'Medium'})
            response_data.append({
                'Category': family,
                'Expected_Uplift': info['uplift'] * 100,
                'SKU_Count': family_counts[family],
                'Response_Level': info['response']
            })

        df_response = pd.DataFrame(response_data)

        fig2 = px.scatter(
            df_response,
            x='SKU_Count',
            y='Expected_Uplift',
            size='SKU_Count',
            color='Response_Level',
            hover_data=['Category'],
            title="Promotion ROI Matrix",
            labels={'Expected_Uplift': 'Uplift (%)', 'SKU_Count': 'SKUs'},
            color_discrete_map={
                'Very High': '#e74c3c',
                'High': '#3498db',
                'Medium': '#f39c12',
                'Low': '#95a5a6'
            }
        )
        fig2.update_layout(height=500, title_font_size=16, font=dict(size=11))

        # Chart 3: Opportunity Calendar
        weekly_data = []
        for occ in promo_system.upcoming_occasions[:21]:
            occ_date = datetime.strptime(occ['date'], "%Y-%m-%d")
            priority_score = {'very_high': 4, 'high': 3, 'medium': 2, 'low': 1}.get(occ.get('priority', 'medium'), 2)
            weekly_data.append({
                'Date': occ['date'],
                'Occasion': occ['occasion'],
                'Priority': occ.get('priority', 'medium').title(),
                'Score': priority_score,
            })

        df_weekly = pd.DataFrame(weekly_data)

        fig3 = px.bar(
            df_weekly,
            x='Date',
            y='Score',
            color='Priority',
            title="3-Week Opportunity Calendar",
            labels={'Score': 'Priority Score'},
            hover_data=['Occasion'],
            color_discrete_map={
                'Very High': '#e74c3c',
                'High': '#3498db',
                'Medium': '#f39c12',
                'Low': '#95a5a6'
            }
        )
        fig3.update_layout(height=450, title_font_size=16, xaxis=dict(tickangle=-45))

        # Summary
        total_items = len(promo_system.items_df)
        high_response_items = sum([row['SKU_Count'] for row in response_data if row['Response_Level'] in ['High', 'Very High']])
        avg_uplift = np.mean([row['Expected_Uplift'] for row in response_data])

        summary_html = f"""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    color: white; padding: 25px; border-radius: 12px; margin-bottom: 20px;'>
            <h2 style='margin: 0 0 15px 0;'>📊 Executive Summary</h2>
            <p style='margin: 5px 0;'><strong>Generated:</strong> {current_date}</p>
        </div>

        <div style='display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin-bottom: 20px;'>
            <div style='background: #3498db; color: white; padding: 20px; border-radius: 8px; text-align: center;'>
                <h3 style='margin: 0; font-size: 32px;'>{total_items:,}</h3>
                <p style='margin: 5px 0 0 0;'>Total SKUs</p>
            </div>
            <div style='background: #e74c3c; color: white; padding: 20px; border-radius: 8px; text-align: center;'>
                <h3 style='margin: 0; font-size: 32px;'>{high_response_items:,}</h3>
                <p style='margin: 5px 0 0 0;'>High-Response SKUs</p>
            </div>
            <div style='background: #2ecc71; color: white; padding: 20px; border-radius: 8px; text-align: center;'>
                <h3 style='margin: 0; font-size: 32px;'>{avg_uplift:.1f}%</h3>
                <p style='margin: 5px 0 0 0;'>Avg Expected Uplift</p>
            </div>
        </div>

        <h3 style='color: #2c3e50; margin: 20px 0 10px 0;'>Top Opportunities</h3>
        """

        for i, row in enumerate(response_data[:5], 1):
            summary_html += f"<p style='margin: 5px 0;'>{i}. <strong>{row['Category']}</strong> - {row['Expected_Uplift']:.0f}% uplift, {row['SKU_Count']} SKUs</p>"

        return fig1, fig2, fig3, summary_html

    except Exception as e:
        return None, None, None, f"❌ Error: {str(e)}"

def enhanced_chatbot(message, history):
    """AI chatbot assistant"""
    try:
        if promo_system.items_df is None:
            return history + [[message, "⚠️ **Data Not Loaded**\n\nPlease upload items.csv to /content/ directory."]]

        message_lower = message.lower()
        response = ""

        if any(word in message_lower for word in ["recommend", "suggestion"]):
            available_families = promo_system.items_df['family'].unique()
            priority_families = ['BEVERAGES', 'MEATS', 'GROCERY I']
            selected = [f for f in priority_families if f in available_families][:3]

            response = "**🎯 Top Recommendations:**\n\n"

            for i, family in enumerate(selected, 1):
                items = promo_system.items_df[promo_system.items_df['family'] == family]
                if len(items) > 0:
                    info = promo_system.family_insights.get(family, {})
                    response += f"**{i}. {family}**\n"
                    response += f"• Best Promo: {info.get('best_promo', 'N/A')}\n"
                    response += f"• Expected Uplift: {info.get('uplift', 0.25):.0%}\n\n"

        elif "uplift" in message_lower or "roi" in message_lower:
            response = "**📈 Expected Uplifts by Category:**\n\n"
            sorted_families = sorted(
                promo_system.family_insights.items(),
                key=lambda x: x[1]['uplift'],
                reverse=True
            )

            for i, (family, info) in enumerate(sorted_families[:5], 1):
                if family in promo_system.items_df['family'].values:
                    response += f"{i}. **{family}**: {info['uplift']:.0%}\n"

        else:
            response = "**💡 I can help you with:**\n"
            response += "• Promotion recommendations\n"
            response += "• Category strategies\n"
            response += "• Expected uplifts and ROI\n"
            response += "• Timing suggestions\n\n"
            response += "**Try asking:** 'Give me recommendations' or 'Show uplifts'"

        history.append([message, response])
        return history

    except Exception as e:
        history.append([message, f"❌ Error: {str(e)}"])
        return history

def get_system_status():
    if promo_system.items_df is not None:
        return f"✅ **System Ready** | {len(promo_system.items_df):,} items loaded from {len(promo_system.items_df['family'].unique())} categories"
    else:
        return "⚠️ **Awaiting Data** | Upload items.csv to /content/ directory"

# Create Gradio Interface
with gr.Blocks(
    title="AI Promotion Optimizer",
    theme=gr.themes.Soft(primary_hue="blue"),
    css="""
    .gradio-container {font-size: 14px;}
    """) as app:

    current_date = datetime.now().strftime("%B %d, %Y")
    data_status = "✅ Ready" if promo_system.items_df is not None else "⚠️ Awaiting Data"

    gr.HTML(f"""
    <div style='text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white; padding: 30px; border-radius: 12px; margin-bottom: 20px;'>
        <h1 style='margin: 0; font-size: 2.5em;'>🚀 AI Promotion Optimizer</h1>
        <p style='font-size: 1.1em; margin: 10px 0 0 0;'>{current_date} | {data_status}</p>
    </div>
    """)

    # System Status
    with gr.Accordion("📊 System Status", open=False):
        status_display = gr.Markdown(value=get_system_status())
        refresh_btn = gr.Button("🔄 Refresh", size="sm")
        refresh_btn.click(fn=lambda: get_system_status(), outputs=[status_display])

    # Tab 1: Recommendations (FIRST)
    with gr.Tab("🎯 Recommendations"):
        gr.Markdown("### Generate Personalized Promotion Recommendations")

        with gr.Row():
            num_recs = gr.Slider(3, 15, value=5, step=1, label="Number of Recommendations")
            get_recs_btn = gr.Button("🎯 Generate Recommendations", variant="primary", size="lg")

        recommendations_output = gr.HTML(value="<p style='text-align: center; padding: 40px; color: #7f8c8d;'>Click <strong>Generate Recommendations</strong> to create personalized promotion strategies.</p>")

        get_recs_btn.click(
            get_detailed_recommendations,
            inputs=[num_recs],
            outputs=[recommendations_output]
        )

    # Tab 2: Analytics (SECOND)
    with gr.Tab("📊 Analytics"):
        gr.Markdown("### Business Intelligence Dashboard")
        analyze_btn = gr.Button("📊 Generate Dashboard", variant="primary", size="lg")

        with gr.Row():
            plot1 = gr.Plot(label="Category Distribution")
            plot2 = gr.Plot(label="ROI Matrix")

        plot3 = gr.Plot(label="Opportunity Calendar")
        analysis_output = gr.HTML(label="Executive Summary")

        analyze_btn.click(
            create_analysis_dashboard,
            inputs=[],
            outputs=[plot1, plot2, plot3, analysis_output]
        )

    # Tab 3: AI Assistant (THIRD)
    with gr.Tab("🤖 AI Assistant"):
        gr.Markdown("### Ask Questions About Your Promotions")
        chatbot = gr.Chatbot(height=450, show_label=False)

        with gr.Row():
            msg = gr.Textbox(
                label="",
                placeholder="Ask about promotions, categories, timing...",
                scale=4,
                lines=1
            )
            submit_btn = gr.Button("Send", variant="primary", scale=1)

        with gr.Row():
            q1 = gr.Button("🎯 Recommendations", size="sm")
            q2 = gr.Button("📈 Uplifts", size="sm")
            q3 = gr.Button("💡 Help", size="sm")
            clear_btn = gr.Button("Clear", size="sm", variant="stop")

        submit_btn.click(enhanced_chatbot, [msg, chatbot], [chatbot])
        submit_btn.click(lambda: "", None, [msg])
        msg.submit(enhanced_chatbot, [msg, chatbot], [chatbot])
        msg.submit(lambda: "", None, [msg])

        q1.click(lambda: enhanced_chatbot("Give me recommendations", []), None, [chatbot])
        q2.click(lambda: enhanced_chatbot("Show uplifts", []), None, [chatbot])
        q3.click(lambda: enhanced_chatbot("Help", []), None, [chatbot])
        clear_btn.click(lambda: [], None, [chatbot])

    # Tab 4: Help & Support (LAST)
    with gr.Tab("❓ Help"):
        gr.Markdown("""
        # 📚 Help & Support

        ## 🚀 Getting Started

        ### 1. Upload Your Data
        - Open the file browser in Colab (📁 icon on the left)
        - Upload your `items.csv` file to `/content/` directory
        - System will automatically detect and load the data

        ### 2. Required CSV Format
        Your CSV file must contain these columns:
        - **`item_nbr`** - Unique item identifier
        - **`family`** - Product category name

        ---

        ## 🎯 Using the Recommendations Tab

        1. **Select Number of Recommendations** - Use the slider (3-15)
        2. **Click "Generate Recommendations"** - AI will create personalized strategies
        3. **Review Each Card** - Contains item details, timing, and implementation tips
        4. **Export Results** - Copy recommendations for your team

        ### What You'll Get:
        - **Item Details** - SKU number, category, promotion type
        - **Timing Strategy** - Best dates and occasions
        - **Performance Predictions** - Expected uplift percentages
        - **Why This Works** - Customer behavior insights
        - **Implementation Tips** - Actionable steps to execute

        ---

        ## 📊 Analytics Dashboard Features

        ### Category Distribution Chart
        Shows your product portfolio across categories

        ### ROI Matrix
        Visualizes promotion potential vs. SKU count

        ### Opportunity Calendar
        3-week view of upcoming promotional occasions

        ---

        ## 🤖 AI Assistant Commands

        Try asking:
        - *"Give me recommendations"*
        - *"Show uplifts for beverages"*
        - *"What are the best promotion types?"*
        - *"When should I promote meats?"*

        ---

        ## 🔧 Troubleshooting

        ### Data Not Loading?
        - ✅ Verify file is named `items.csv`
        - ✅ Check it's in `/content/` directory
        - ✅ Confirm columns: `item_nbr` and `family`
        - ✅ Click "Refresh" in System Status

        ### No Recommendations Appearing?
        - ✅ Ensure data is loaded (check System Status)
        - ✅ Click "Generate Recommendations" button
        - ✅ Wait 2-3 seconds for processing

        ### Charts Not Displaying?
        - ✅ Click "Generate Dashboard" button
        - ✅ Scroll down to view all charts
        - ✅ Refresh the page if needed

        ---

        ## 📋 Supported Categories

        Our system recognizes these product families:
        - **BEVERAGES** - Drinks, juices, sodas
        - **MEATS** - Fresh and frozen meats
        - **GROCERY I** - Packaged goods
        - **PRODUCE** - Fresh fruits and vegetables
        - **DAIRY** - Milk, cheese, yogurt
        - **BREAD/BAKERY** - Fresh baked goods
        - **FROZEN FOODS** - Frozen meals and items
        - **CLEANING** - Household cleaning products
        - **PERSONAL CARE** - Health and hygiene
        - **BEAUTY** - Cosmetics and skincare
        - **HOME CARE** - Home maintenance products
        - **LIQUOR,WINE,BEER** - Alcoholic beverages
        - **AUTOMOTIVE** - Car care products

        ---

        ## 🎨 Promotion Types Available

        - **BOGO** - Buy One Get One
        - **Flash Sale** - 24-48 hour urgency
        - **Percentage Off** - 20-40% discounts
        - **Volume Discount** - Bulk purchase savings
        - **Bundle Deal** - Multi-item packages
        - **Limited Time Offer** - Exclusive periods

        ---

        ## 📈 Understanding Metrics

        ### Uplift Percentage
        Expected increase in sales during promotion period
        - **40%+** = Very High Response
        - **30-40%** = High Response
        - **20-30%** = Medium Response
        - **Below 20%** = Low Response

        ### Response Level
        Customer engagement and purchase likelihood
        - Helps prioritize which categories to promote

        ### Timing Advantage
        Strategic reasoning for recommended dates
        - Based on customer behavior patterns

        ---

        ## 💾 File Upload Methods

        ### Method 1: Direct Upload (Recommended)
        ```
        1. Click 📁 folder icon in Colab sidebar
        2. Drag and drop items.csv
        3. File appears in /content/
        ```

        ### Method 2: Google Drive
        ```
        1. Mount Google Drive in Colab
        2. Copy items.csv to /content/drive/MyDrive/
        3. System auto-detects file
        ```

        ---

        ## 🆘 Need More Help?

        **System Status Not Green?**
        - Re-upload your CSV file
        - Check file format and columns
        - Restart the Colab runtime

        **Questions About Results?**
        - Use the AI Assistant tab
        - Ask specific questions about categories
        - Request explanations for recommendations

        ---

        ## 📌 Best Practices

        1. **Generate Fresh Recommendations Weekly** - Market conditions change
        2. **Review All Metrics** - Don't just look at uplift percentages
        3. **Consider Multiple Categories** - Diversify your promotion calendar
        4. **Plan Ahead** - Recommendations include future dates
        5. **Monitor Results** - Track actual vs. predicted performance

        ---

        ## ⚡ Quick Tips

        - 💡 Higher uplift = Better promotion opportunity
        - 💡 Weekend promotions work best for beverages and meats
        - 💡 Month-start promotions align with payday spending
        - 💡 Flash sales create urgency for perishables
        - 💡 Bundle deals increase basket size

        ---

        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    color: white; padding: 20px; border-radius: 8px; text-align: center; margin-top: 30px;'>
            <h3 style='margin: 0;'>🚀 AI Promotion Optimizer v2.0</h3>
            <p style='margin: 10px 0 0 0; font-size: 14px;'>© 2025 | Built with AI Intelligence</p>
        </div>
        """)

    # Footer
    gr.HTML("""
    <div style='text-align: center; margin-top: 30px; padding: 15px;
                background: #f8f9fa; border-radius: 8px; border-top: 3px solid #667eea;'>
        <p style='margin: 0; color: #6c757d; font-size: 13px;'>
            🚀 <strong>AI Promotion Optimizer</strong> | Version 2.0 | Powered by Advanced Analytics
        </p>
    </div>
    """)

# Launch Application
if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 AI PROMOTION OPTIMIZER v2.0")
    print("="*60)
    print(f"\n{initial_load_status}\n")
    print("="*60)
    print("\n🌐 Launching interface...")
    print("✨ Features loaded:")
    print("   • Smart Recommendations with Card Layout")
    print("   • Advanced Analytics Dashboard")
    print("   • AI-Powered Assistant")
    print("   • Comprehensive Help System")
    print("\n" + "="*60 + "\n")

    app.launch(share=True, debug=True)

✅ Successfully loaded 4100 items from 33 categories!


/tmp/ipython-input-1659532111.py:569: UserWarning:

You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.




🚀 AI PROMOTION OPTIMIZER v2.0

✅ Successfully loaded 4100 items from 33 categories!


🌐 Launching interface...
✨ Features loaded:
   • Smart Recommendations with Card Layout
   • Advanced Analytics Dashboard
   • AI-Powered Assistant
   • Comprehensive Help System


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2f2cc0eefaf1d33c99.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2f2cc0eefaf1d33c99.gradio.live
